# Data preprocessing

In [1]:
#load dataset
from datasets import load_dataset
ds = load_dataset("json", data_files="../data/unprocessed/arxiv-metadata-oai-snapshot.json")

c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed'],
        num_rows: 2292057
    })
})


In [3]:
import re
import json
import os
from pathlib import Path
from collections import Counter
from datasets import load_dataset

In [4]:
# Thiết lập đường dẫn
DATA_DIR = Path("../data")
UNPROCESSED_DIR = DATA_DIR / "unprocessed"
PROCESSED_DIR = DATA_DIR / "processed"

# Tạo thư mục nếu chưa tồn tại
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# Load dataset
print("Đang tải dataset...")
ds = load_dataset("json", data_files=str(UNPROCESSED_DIR / "arxiv-metadata-oai-snapshot.json"))
data = ds['train']
print(f"✓ Tổng số mẫu: {len(data):,}")
print(f"✓ Các trường: {data.column_names}")

Đang tải dataset...
✓ Tổng số mẫu: 2,292,057
✓ Các trường: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed']


In [ ]:
# Hàm tiền xử lý abstract
def preprocess_abstract(text):
    """Tiền xử lý abstract: giống hệt đoạn xử lý mẫu đã cho"""
    if not text or not isinstance(text, str):
        return None

    # Remove \n characters in the middle and leading/trailing spaces
    text = text.strip().replace("\n", " ")

    # Remove special characters
    text = re.sub(r'[^\w\s]', '', text)

    # Remove digits
    text = re.sub(r'\d+', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Convert to lower case
    text = text.lower()

    return text

# Hàm rút gọn category
def extract_category(categories):
    """Trích xuất category chính"""
    if not categories or not isinstance(categories, str):
        return None
    
    first_category = categories.strip().split()[0]
    main_category = first_category.split('.')[0]
    
    return main_category

In [7]:
# Xử lý và GHI TRỰC TIẾP vào file (JSON Lines format)
print("\n" + "="*60)
print("BẮT ĐẦU XỬ LÝ VÀ GHI FILE")
print("="*60)

output_file = PROCESSED_DIR / "data_arxiv_preprocessed.jsonl"  # Đổi extension thành .jsonl
skipped_stats = {
    'missing_abstract': 0,
    'missing_categories': 0,
    'invalid_length': 0,
    'processing_error': 0
}
category_counter = Counter()
processed_count = 0
# MỞ FILE ĐỂ GHI TỪNG DÒNG
with open(output_file, 'w', encoding='utf-8') as f:
    for i, sample in enumerate(data):
        try:
            # Lấy dữ liệu
            abstract = sample.get('abstract', '')
            categories = sample.get('categories', '')
            
            # Kiểm tra dữ liệu thiếu
            if not abstract:
                skipped_stats['missing_abstract'] += 1
                continue
            if not categories:
                skipped_stats['missing_categories'] += 1
                continue
            
            # Tiền xử lý
            cleaned_abstract = preprocess_abstract(abstract)
            main_category = extract_category(categories)
            
            # Bỏ qua nếu xử lý thất bại
            if not cleaned_abstract:
                skipped_stats['invalid_length'] += 1
                continue
            if not main_category:
                skipped_stats['processing_error'] += 1
                continue
            
            # Tạo dict cho mẫu này
            processed_sample = {
                "text": cleaned_abstract,
                "label": main_category
            }
            
            # GHI MỘT DÒNG JSON (không có indent, không có dấu phẩy)
            f.write(json.dumps(processed_sample, ensure_ascii=False) + '\n')
            
            # Đếm
            processed_count += 1
            category_counter[main_category] += 1
            
            # In tiến trình
            if (i + 1) % 50000 == 0:
                print(f"  Đã xử lý: {i + 1:,} / {len(data):,} ({(i+1)/len(data)*100:.1f}%)")
                print(f"  Đã ghi: {processed_count:,} mẫu")
        
        except Exception as e:
            skipped_stats['processing_error'] += 1
            if skipped_stats['processing_error'] <= 5:
                print(f"  ⚠ Lỗi tại mẫu {i}: {str(e)[:100]}")

print(f"\n✓ Đã ghi xong file: {output_file}")


BẮT ĐẦU XỬ LÝ VÀ GHI FILE
  Đã xử lý: 50,000 / 2,292,057 (2.2%)
  Đã ghi: 50,000 mẫu
  Đã xử lý: 100,000 / 2,292,057 (4.4%)
  Đã ghi: 100,000 mẫu
  Đã xử lý: 150,000 / 2,292,057 (6.5%)
  Đã ghi: 150,000 mẫu
  Đã xử lý: 200,000 / 2,292,057 (8.7%)
  Đã ghi: 200,000 mẫu
  Đã xử lý: 250,000 / 2,292,057 (10.9%)
  Đã ghi: 250,000 mẫu
  Đã xử lý: 300,000 / 2,292,057 (13.1%)
  Đã ghi: 300,000 mẫu
  Đã xử lý: 350,000 / 2,292,057 (15.3%)
  Đã ghi: 350,000 mẫu
  Đã xử lý: 400,000 / 2,292,057 (17.5%)
  Đã ghi: 400,000 mẫu
  Đã xử lý: 450,000 / 2,292,057 (19.6%)
  Đã ghi: 450,000 mẫu
  Đã xử lý: 500,000 / 2,292,057 (21.8%)
  Đã ghi: 500,000 mẫu
  Đã xử lý: 550,000 / 2,292,057 (24.0%)
  Đã ghi: 550,000 mẫu
  Đã xử lý: 600,000 / 2,292,057 (26.2%)
  Đã ghi: 600,000 mẫu
  Đã xử lý: 650,000 / 2,292,057 (28.4%)
  Đã ghi: 650,000 mẫu
  Đã xử lý: 700,000 / 2,292,057 (30.5%)
  Đã ghi: 700,000 mẫu
  Đã xử lý: 750,000 / 2,292,057 (32.7%)
  Đã ghi: 750,000 mẫu
  Đã xử lý: 800,000 / 2,292,057 (34.9%)
  Đã ghi:

In [8]:
# In thống kê
print("\n" + "="*60)
print("THỐNG KÊ XỬ LÝ")
print("="*60)
print(f"✓ Tổng mẫu đầu vào:        {len(data):,}")
print(f"✓ Mẫu xử lý thành công:    {processed_count:,}")
print(f"✗ Mẫu bị loại bỏ:          {sum(skipped_stats.values()):,}")
print(f"\nChi tiết mẫu bị loại:")
print(f"  - Thiếu abstract:        {skipped_stats['missing_abstract']:,}")
print(f"  - Thiếu categories:      {skipped_stats['missing_categories']:,}")
print(f"  - Độ dài không hợp lệ:   {skipped_stats['invalid_length']:,}")
print(f"  - Lỗi xử lý khác:        {skipped_stats['processing_error']:,}")



THỐNG KÊ XỬ LÝ
✓ Tổng mẫu đầu vào:        2,292,057
✓ Mẫu xử lý thành công:    2,292,057
✗ Mẫu bị loại bỏ:          0

Chi tiết mẫu bị loại:
  - Thiếu abstract:        0
  - Thiếu categories:      0
  - Độ dài không hợp lệ:   0
  - Lỗi xử lý khác:        0


In [ ]:
# Thống kê categories
print("\n" + "="*60)
print("PHÂN BỐ CATEGORIES (Top 38)")
print("="*60)
for cat, count in category_counter.most_common(38):
    percentage = (count / processed_count) * 100
    bar = "█" * int(percentage)
    print(f"{cat:10s} | {count:7,} ({percentage:5.2f}%) {bar}")

print(f"\nTổng số categories:        {len(category_counter)}")


PHÂN BỐ CATEGORIES (Top 15)
math       | 461,568 (20.14%) ████████████████████
cs         | 431,766 (18.84%) ██████████████████
cond-mat   | 297,127 (12.96%) ████████████
astro-ph   | 285,798 (12.47%) ████████████
physics    | 163,944 ( 7.15%) ███████
hep-ph     | 127,013 ( 5.54%) █████
hep-th     | 101,262 ( 4.42%) ████
quant-ph   |  99,689 ( 4.35%) ████
gr-qc      |  59,456 ( 2.59%) ██
stat       |  43,860 ( 1.91%) █
eess       |  39,346 ( 1.72%) █
nucl-th    |  32,059 ( 1.40%) █
math-ph    |  30,347 ( 1.32%) █
q-bio      |  25,996 ( 1.13%) █
hep-ex     |  21,951 ( 0.96%) 
nlin       |  18,280 ( 0.80%) 
hep-lat    |  17,329 ( 0.76%) 
nucl-ex    |  11,132 ( 0.49%) 
q-fin      |  10,223 ( 0.45%) 
econ       |   5,708 ( 0.25%) 
chao-dyn   |   1,770 ( 0.08%) 
alg-geom   |   1,209 ( 0.05%) 
q-alg      |   1,177 ( 0.05%) 
cmp-lg     |     894 ( 0.04%) 
solv-int   |     844 ( 0.04%) 
dg-ga      |     562 ( 0.02%) 
patt-sol   |     452 ( 0.02%) 
funct-an   |     320 ( 0.01%) 
adap-org   |  

In [10]:
# Xuất thống kê
stats_file = PROCESSED_DIR / "category_stats.json"
with open(stats_file, 'w', encoding='utf-8') as f:
    json.dump({
        'total_samples': processed_count,
        'total_categories': len(category_counter),
        'skipped_stats': skipped_stats,
        'category_distribution': dict(category_counter.most_common())
    }, f, ensure_ascii=False, indent=2)

print(f"\n✓ Đã xuất thống kê vào: {stats_file}")


✓ Đã xuất thống kê vào: ..\data\processed\category_stats.json


In [11]:
print("\n" + "="*60)
print("KIỂM TRA LOAD FILE VỚI DATASETS")
print("="*60)

# Load bằng datasets (giống như file gốc)
ds_processed = load_dataset("json", data_files=str(output_file))
print(f"✓ Load thành công!")
print(f"✓ Số mẫu: {len(ds_processed['train']):,}")
print(f"✓ Các trường: {ds_processed['train'].column_names}")


KIỂM TRA LOAD FILE VỚI DATASETS


Generating train split: 2292057 examples [00:08, 259366.91 examples/s]


✓ Load thành công!
✓ Số mẫu: 2,292,057
✓ Các trường: ['text', 'label']


In [2]:
from datasets import load_dataset
ds = load_dataset("json", data_files="../data/unprocessed/arxiv-metadata-oai-snapshot.json")

DatasetDict({
    train: Dataset({
        features: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed'],
        num_rows: 2292057
    })
})